In [ ]:
%load_ext autoreload
%autoreload 2

import edunn as nn
import numpy as np

# Activation functions

A `neural network` is a model that combines the transformations of several layers, such as those of Linear Regression or Logistic Regression. But to define a network we must use _activation functions_, which let us apply nonlinear transformations to the data. Otherwise, combining two Linear Regression layers, for example, is equivalent to having a single one, since it will only be able to represent a straight line. In that case our network would not be any more powerful, only less efficient.

There are different types of activation functions: we are going to look at three of the most common ones, `relu`, `sigmoid` (or `logistic`) and `tanh` (hyperbolic tangent). Each one has different properties:

* `relu`: it is efficient to compute both its output and its derivative, and although its nonlinearity is simple it still serves to give a network greater approximation power. On the other hand, its output lies in the range $[0,\infty)$, which makes it unsuitable for some tasks
* `sigmoid`: it is efficient, though not as much as `relu`, and since its range is $(0,1)$ it can encode a value that may be interpreted as a probability
* `tanh`: it is similar to `sigmoid`, but its range is $(-1,1)$, so it can encode both positive and negative values and thereby build representations that feed those signs back into the network.

<div style="background-color: white;">
<img src="img/relu.png" width="25%" style="display:inline-block"> <img src="img/sigmoid.png" width="25%" style="display:inline-block"> <img src="img/tanh.png" width="25%" style="display:inline-block">
</div>

# ReLU Layer

The `ReLU` (Rectified Linear Unit) function is extremely simple.

<div style="background-color: white;">
<img src="img/relu.png" width=50%>
</div>

Formally,

$$ReLU(x) = \begin{cases}
    0 & \text{if } x \le 0 \\ 
    x & \text{if }  x > 0
\end{cases}
$$

More compactly, though a little harder to understand, we can write `ReLU` as:
$$ReLU(x) = max(0,x)$$

In terms of code, computing `relu` for a single value is simple:

````python
def relu(x:float):
    if x>0:
        return x
    else:
        return 0
````

Keep in mind, though, that here you will receive a tensor of values. Even so, `ReLU`, like the other activation functions, is easy to compute because it is applied _element by element_. That is, `ReLU` of a vector is equivalent to applying `ReLU` to each one of its values, that is:

$$ReLU( (-2,4,7)=( ReLU(-2), ReLU(4), ReLU(7)) = (0,4,7)$$

For a matrix, or a tensor in general, you proceed in the same way.

Implement the `forward` method of the `ReLU` class in the file `edunn/models/activations.py`


In [ ]:
x = np.array([[3.5,-7.2,5.3],
             [-3.5,7.2,-5.3]])

layer=nn.ReLU()
y = np.array([[3.5,0,5.3],
             [0,7.2,0]])
nn.utils.check_same(y,layer.forward(x))

# plot values
nn.plot.plot_activation_function(layer)

# Backward method of `ReLU`

Fortunately, `ReLU`'s `backward` method is simple. Working case by case, we can see that if the forward pass is:

$$ReLU(x) = \begin{cases}
    0 & \text{if } x \le 0 \\ 
    x & \text{if }  x > 0
\end{cases}
$$

Then we can differentiate each case separately, giving:
$$\frac{d ReLU(x)}{dx} = \begin{cases}
    0 & \text{if } x \le 0 \\ 
    1 & \text{if }  x > 0
\end{cases}
$$

At $x=0$ the derivative of $ReLU$ is not actually defined; nevertheless, in this case we can redefine that derivative as 0 (or 1), and it will not really affect the optimization.

In [ ]:
from edunn.utils import check_gradient


# Number of random examples and their size, used to generate
# samples of x and check the derivatives
samples = 100
input_shape=(5,2)

# Check the derivatives of a ReLU function
layer=nn.ReLU()
check_gradient.common_layer(layer,input_shape,samples=samples)

nn.plot.plot_activation_function(nn.ReLU(),backward=True)

# Sigmoid Layer

The `sigmoid` function (sigmoidal, also known as the logistic function) maps any value into the interval $(0,1)$. 

<div style="background-color: white;">
<img src="img/sigmoid.png" width=50%>
</div>

Its definition is:

$$
Sigmoid(x)= \frac{1}{1+e^{-x}}
$$

So, for example, we have:

$$ sigmoid(0)=\frac{1}{1+1}=\frac{1}{2}=0.5$$
$$ sigmoid(1)=\frac{1}{1+e^{-1}}=\frac{1}{1+0.36}=0.73$$
$$ sigmoid(-1)=\frac{1}{1+e^{-(-1)}}=\frac{1}{1+2.71}=0.26$$

As we can see, the function is balanced at the value $0$, where the output is $0.5$; values greater than $0$ produce larger outputs, and vice versa.


Implement the `forward` method of the `Sigmoid` class in the file `edunn/models/activations.py`


In [ ]:
x = np.array([[0,1,-1]])

layer=nn.Sigmoid()
y = np.array([[0.5,0.73105858,0.26894142]])
nn.utils.check_same(y,layer.forward(x),tol=1e-6)
nn.plot.plot_activation_function(layer)

# Backward method of `Sigmoid`

The derivatives for `Sigmoid`'s `backward` method are simple, but we will also want to rewrite them in a simplified form that makes them more efficient to compute.

$$
Sigmoid(x)= \frac{1}{1+e^{-x}}
$$

$$
\begin{aligned}
\frac{d Sigmoid(x)}{dx} 
&= \frac{d \frac{1}{1+e^{-x}}}{dx} =\frac{d (1+e^{-x})^{-1}}{dx} \\
&= \frac{d (1+e^{-x})^{-1}}{d(1+e^{-x})} \frac{d (1+e^{-x})}{dx} & \text{(regla de la cadena con $g(x)=1+e^{-x}$)} \\
&= \frac{d  (1+e^{-x})^{-1} }{d(1+e^{-x})} (-e^{-x}) & \text{(derivada de $1+e^{-x}$)} \\
&=  -(1+e^{-x})^{-2} (-e^{-x}) & \text{(derivada de $g(x)^{-1}=-g(x)^{-2}$)} \\
&=  (1+e^{-x})^{-2} e^{-x}\\
&=  Sigmoid(x)^2 e^{-x}\\
\end{aligned}
$$

At this point we have a formula for the derivative of $Sigmoid$, but as we said before, we are going to turn it into a simpler formula that only uses the original $Sigmoid$ value. That way we avoid recomputing exponentiations or divisions, which are computationally more expensive. To do so, we start from the second-to-last line of the derivation above:

<!-- $$
\begin{aligned}
\frac{d Sigmoid(x)}{dx} &=  (1+e^{-x})^{-2} e^{-x}\\
&=  Sigmoid(x)^2 e^{-x}\\
&=  Sigmoid(x)^2 (1-1+e^{-x})\\
&=  Sigmoid(x)^2 (1-Sigmoid(x)^{-1})\\
&=  Sigmoid(x) Sigmoid(x) (1-Sigmoid(x)^{-1})\\
&=  Sigmoid(x) [Sigmoid(x) (1-Sigmoid(x)^{-1})]\\
&=  Sigmoid(x) [Sigmoid(x) * 1- Sigmoid(x) Sigmoid(x)^{-1}]\\
&=  Sigmoid(x) (Sigmoid(x) -1)\\
\end{aligned}
$$ -->

<!-- https://www.desmos.com/calculator/ekqs9htkml -->

$$
\begin{aligned}
\frac{d Sigmoid(x)}{dx} &=  (1+e^{-x})^{-2} e^{-x}\\
&= \frac{e^{-x}}{(1 + e^{-x})^2} \\
&= \frac{1}{1 + e^{-x}} \frac{e^{-x}}{1 + e^{-x}} \\
&= \frac{1}{1 + e^{-x}} \frac{(1 + e^{-x}) - 1}{1 + e^{-x}} \\
&= \frac{1}{1 + e^{-x}} \left(1 - \frac{1}{1 + e^{-x}}\right) \\
&= Sigmoid(x) (1 - Sigmoid(x))
\end{aligned}
$$

As we can see, this final formula $Sigmoid'(x) = Sigmoid(x)  (1-Sigmoid(x))$ tells us that if we store the value of $Sigmoid(x)$ during the `forward` pass, then in the `backward` pass the derivative is computed simply as `y*(1-y)`, which only requires one vector addition and one vector multiplication.

Implement the `backward` method of the `Sigmoid` class and check it with the following code:

In [ ]:
from edunn.utils import check_gradient


# Number of random examples and their size, used to generate
# samples of x and check the derivatives
samples = 100
input_shape=(5,2)

# Check the derivatives of a Sigmoid function
layer=nn.Sigmoid()
check_gradient.common_layer(layer,input_shape,samples=samples)

nn.plot.plot_activation_function(layer,backward=True)

# `TanH` Layer

The `tanh` (hyperbolic tangent) function maps any value into the interval $(-1,1)$. 

<div style="background-color: white;">
<img src="img/tanh.png" width=35%>
</div>

$tanh$ is a [hyperbolic](https://en.wikipedia.org/wiki/Hyperbolic_functions) geometric function. Just as $tan(x)=\frac{sen(x)}{cos(x)}$, $tanh$ is defined in terms of the hyperbolic cosine and sine:

$$
tanh(x)= \frac{senh(x)}{cosh(x)} = \frac{e^x-e^{-x}}{e^x+e^{-x}}
$$

For example:

$$ 
\begin{aligned}
tanh(0)  &= \frac{1-1}{1+1} &&= \frac{0}{2} &&= 0 \\
tanh(1)  &= \frac{e-e^{-1}}{e+e^{-1}} &&= \frac{2.35}{3.08} &&= 0.76 \\
tanh(-1) &= \frac{e^{-1}-e}{e^{-1}+e} &&= \frac{-2.35}{3.08} &&= -0.76
\end{aligned} 
 $$

The function is balanced at the value $0$, where the output is $0$; values greater than $0$ produce larger outputs, and vice versa, which is why $tanh$ is an odd function ($tanh(x)=-tanh(-x)$).

$TanH$ is very similar to the $Sigmoid$ function. In fact, in the plot below we can see that $TanH$ is simply $Sigmoid$, but:

* Multiplied by two (to turn the range $(0,1)$ into the range $(0,2)$)
* With 1 subtracted from it (to turn the range $(0,2)$ into the range $(-1,1)$)
* With $x$ multiplied by 2, so that the two curves match

<div style="background-color: white;">
<img src="img/tanh_sigmoid.png" width=35%> 
</div>

Indeed, $tanh$ [can be defined](http://facundoq.github.io/edunn/material/sigmoid_tanh) in terms of $Sigmoid$:
$$
tanh(x) = sigmoid(2x)*2-1
$$

_Errata_ for the previous note:

$$ 
\begin{aligned}
tanh(x) &= \frac{e^x-e^{-x}}{e^x+e^{-x}} \\
&= \frac{e^x}{e^x} \frac{e^x-e^{-x}}{e^x+e^{-x}} \\
&= \frac{e^{2x}-1}{e^{2x}+1} \\
&= \frac{e^{w}-1}{e^{w}+1} & \text{(sea $w=2x$)} \\
&= \frac{-1+e^{w}+1-1}{1+e^{w}} \\
&= \frac{-2+e^{w}+1}{1+e^{w}} \\
&= \frac{-2}{1+e^{w}} +  \frac{e^{w}+1}{1+e^{w}} \\
&= -2 Sigmoid(-w) + 1 \\
&= 2 Sigmoid(w) - 1 & \text{(dado que $tanh(x)\stackrel{\text{impar}}{=}-tanh(-x)$)} \\
&= 2 Sigmoid(2x) - 1 \\
\end{aligned} 
$$

This form will be much more convenient for the implementation, since we can reuse the `Sigmoid` layer for both the forward and the backward pass.

Implement the `forward` method of the `TanH` class in the file `edunn/models/activations.py`  _using the `Sigmoid` layer_ (we help you out by already defining a variable for it). 



In [ ]:
x = np.array([[0,0.5,-0.5]])

layer=nn.TanH()
y = np.array([[ 0., 0.46211716, -0.46211716]])
nn.utils.check_same(y,layer.forward(x),tol=1e-6)

nn.plot.plot_activation_function(layer)


# Backward method of `TanH`

The derivatives for `TanH`'s `backward` method can be obtained from those of `Sigmoid`. Since

$$
tanh(x) = sigmoid(2x)*2-1
$$

Then:

$$
tanh'(x) = (sigmoid'(2x)*2)*2 = sigmoid'(2x)*4
$$

That is, the derivative of $tanh$ consists simply of multiplying the derivative of $sigmoid$ by four.

Implement the `backward` method of the `TanH` class and check it with the following code:



In [ ]:
from edunn.utils import check_gradient


# Number of random examples and their size, used to generate
# samples of x and check the derivatives
samples = 100
input_shape=(5,2)

# Check the derivatives of a Sigmoid function
layer=nn.TanH()
check_gradient.common_layer(layer,input_shape,samples=samples)

nn.plot.plot_activation_function(layer,backward=True)